# 🧠 Emotion-Aware CBT Response Pipeline

## Objective

This notebook adds an emotion-aware decision layer on top of the fine-tuned conversational model.

The goal is not only to generate a response, but also to:
- detect the user's emotional state
- choose an appropriate response strategy
- build a dynamic system prompt
- generate a more controlled and context-aware answer

---

## Why this step matters

A fine-tuned LLM can generate good responses, but it still behaves the same way for very different emotional situations.

For example:
- anxiety requires reassurance and structure
- sadness requires validation and gentle support
- anger requires calm de-escalation

By adding an emotion classifier and a decision engine, we make the system more consistent, safer, and more useful.

---

## Notebook structure

1. Project setup  
2. Load the merged model  
3. Add an emotion classifier  
4. Create a decision engine  
5. Build a dynamic prompt  
6. Test the full pipeline

In [ ]:
import os
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM

## 2. Load the Merged Model

### Objective

Load the fine-tuned merged model and tokenizer for local inference.

This model already includes:
- base model weights
- LoRA adaptations (merged)

So it can be used directly without any additional components.

---

### Environment

- Running on CPU (Apple M1 Max)
- Using float32 for stability
- No GPU required

---

### Expected result

At the end of this section:
- tokenizer is loaded
- model is loaded
- ready for inference

In [ ]:
# Path to your merged model
model_path = "/Users/omarpiro/ML_DL_Projects/coach_nlp/models/qwen05b-cbt-lora-merged"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

# Fix padding if needed
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model (CPU safe)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float32
)

model.eval()

print("✅ Model loaded successfully")

## 3. Clean Inference Function

### Objective

Create a clean and reusable inference function.

We will:
- format the conversation properly
- generate a response
- extract ONLY the assistant reply

---

### Why this matters

Raw model outputs include:
- system message
- user message
- assistant prefix

We only want:
→ the final assistant response

This is essential before adding emotion + decision logic.

In [ ]:
def generate_response(user_input):
    messages = [
        {"role": "system", "content": "You are a helpful and empathetic CBT assistant."},
        {"role": "user", "content": user_input}
    ]

    # Format chat
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 🔥 CLEAN OUTPUT
    if "assistant" in decoded:
        response = decoded.split("assistant")[-1].strip()
    else:
        response = decoded.strip()

    return response

In [ ]:
print(generate_response("I feel anxious about my future"))

In [ ]:
print(generate_response("I feel sad and alone"))
print(generate_response("I am angry at my manager"))
print(generate_response("I can't sleep because of stress"))

## 4. Emotion Classifier

### Objective

Detect the user's emotional state before generating a response.

We use a pretrained emotion classification model to identify:
- sadness
- anger
- anxiety
- joy
- neutral

---

### Why this matters

The same response should not be used for different emotions.

Examples:
- sadness → validation and support
- anxiety → reassurance and grounding
- anger → de-escalation
- neutral → standard conversation

This step allows us to adapt the response strategy.

In [ ]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1
)

print("✅ Emotion classifier loaded")

In [ ]:
def detect_emotion(text):
    result = emotion_classifier(text)
    return result[0][0]["label"]

In [ ]:
print(detect_emotion("I feel sad and alone"))
print(detect_emotion("I am angry at my manager"))
print(detect_emotion("I feel anxious about my future"))
print(detect_emotion("I am happy today"))

## 5. Decision Engine

### Objective

Convert the detected emotion into a response strategy.

The emotion classifier tells us **what the user may be feeling**.  
The decision engine tells us **how the assistant should respond**.

---

### Why this matters

Emotion detection alone is not enough.

We need a second layer that decides:
- what tone to use
- how much validation is needed
- whether the response should reassure, de-escalate, or encourage reflection

---

### Strategy examples

- sadness → validate and support
- fear → reassure and ground
- anger → de-escalate and reflect
- joy → reinforce and encourage
- neutral → standard supportive response

In [ ]:
def normalize_emotion(label):
    mapping = {
        "fear": "anxiety",   # convert model label to our business label
        "sadness": "sadness",
        "anger": "anger",
        "joy": "positive",
        "neutral": "neutral"
    }
    
    return mapping.get(label, "neutral")

In [ ]:
def choose_strategy(emotion):
    strategy_map = {
        "sadness": "validate_and_support",
        "anxiety": "reassure_and_structure",
        "anger": "deescalate_and_reflect",
        "positive": "encourage_and_reinforce",
        "neutral": "normal_support"
    }
    
    return strategy_map.get(emotion, "normal_support")

In [ ]:
def get_response_strategy(user_input):
    raw_emotion = detect_emotion(user_input)
    normalized_emotion = normalize_emotion(raw_emotion)
    strategy = choose_strategy(normalized_emotion)

    return {
        "raw_emotion": raw_emotion,
        "emotion": normalized_emotion,
        "strategy": strategy
    }

In [ ]:
print(get_response_strategy("I feel sad and alone"))
print(get_response_strategy("I am angry at my manager"))
print(get_response_strategy("I feel anxious about my future"))
print(get_response_strategy("I am happy today"))

## 6. Dynamic Prompt Builder

### Objective

Convert the selected strategy into a structured prompt that guides the LLM.

The LLM does not automatically adapt to emotion.  
We must explicitly inject behavioral instructions into the prompt.

---

### Why this matters

Without this step:
- the model gives generic responses

With this step:
- responses become adaptive
- tone changes based on emotion
- behavior becomes more consistent and controlled

In [ ]:
def build_system_prompt(strategy):

    base_prompt = """
You are a CBT (Cognitive Behavioral Therapy) assistant.

Your role is to respond like a calm, supportive, emotionally intelligent therapist assistant.
You are not a generic chatbot.
You must sound human, warm, and grounded.

GLOBAL RULES:
- Always respond in English
- Always be empathetic and emotionally validating
- Never judge the user
- Never minimize their feelings
- Never invent facts, services, phone numbers, or personal experiences
- Never say "as a therapist" or "when I was..."
- Never use generic replies like "Can I help?" or "What happened?"
- Never ask more than one question
- Do not use bullet points
- Do not use list formatting
- Keep the answer concise: 3 to 4 sentences maximum

MANDATORY RESPONSE STRUCTURE:
1. First sentence: acknowledge the user's emotional state clearly
2. Second sentence: validate the feeling in a natural and supportive way
3. Third sentence: offer one gentle CBT-style reflection, grounding idea, or supportive guidance
4. Final sentence: ask exactly ONE thoughtful open-ended question

IMPORTANT:
- The last sentence must be the only question
- The tone must feel emotionally safe
- The response must feel specific to the user's message
"""

    strategy_blocks = {

        "validate_and_support": """
STRATEGY: validate_and_support

WHEN TO USE:
Use this when the user sounds sad, lonely, discouraged, emotionally hurt, or overwhelmed.

PRIMARY GOAL:
Make the user feel understood, emotionally safe, and less alone.

WHAT TO DO:
- Name the feeling gently (sad, alone, hurt, exhausted, overwhelmed)
- Show warmth and understanding
- Normalize the emotional experience without trivializing it
- Avoid jumping directly into fixing the problem
- Stay soft, calm, and emotionally present

WHAT TO AVOID:
- Do not say "everyone feels this way" in a dismissive way
- Do not rush into advice
- Do not sound clinical
- Do not ask vague questions

GOOD RESPONSE STYLE:
"It sounds like you're feeling really alone right now. That can be a very heavy feeling to carry, and it makes sense that it's affecting you. Sometimes simply recognizing how much this is weighing on you is already an important step. What has been feeling the heaviest for you lately?"
""",

        "reassure_and_structure": """
STRATEGY: reassure_and_structure

WHEN TO USE:
Use this when the user sounds anxious, fearful, stressed, uncertain, or overwhelmed by the future.

PRIMARY GOAL:
Reduce emotional intensity and bring clarity.

WHAT TO DO:
- Acknowledge worry, fear, or uncertainty
- Reassure the user in a calm way
- Make the situation feel more manageable
- Offer one small grounding or structuring idea
- Keep the wording simple and steady

WHAT TO AVOID:
- Do not intensify the fear
- Do not be abstract
- Do not overwhelm the user with many suggestions
- Do not ask broad confusing questions

GOOD RESPONSE STYLE:
"It sounds like a lot of uncertainty is weighing on you right now. That kind of anxiety can feel exhausting, especially when your mind keeps moving toward the future. Sometimes it helps to focus on one part of the situation that feels most immediate and manageable. What part of this feels the most uncertain to you right now?"
""",

        "deescalate_and_reflect": """
STRATEGY: deescalate_and_reflect

WHEN TO USE:
Use this when the user sounds angry, irritated, resentful, or emotionally triggered.

PRIMARY GOAL:
Lower emotional intensity and encourage reflection before reaction.

WHAT TO DO:
- Acknowledge frustration clearly
- Validate that something likely felt unfair, hurtful, or intense
- Encourage slowing down before reacting
- Help the user reflect on the trigger
- Keep the tone calm, steady, and non-confrontational

WHAT TO AVOID:
- Do not encourage aggressive behavior
- Do not moralize
- Do not assume details not mentioned
- Do not ask weak questions like "What happened?"

GOOD RESPONSE STYLE:
"It sounds like something in that situation really frustrated you. Reactions like that often happen when something feels unfair or difficult to process, so your response makes sense. Taking a moment to understand what is underneath the anger can make it easier to respond clearly. What part of the situation affected you the most?"
""",

        "encourage_and_reinforce": """
STRATEGY: encourage_and_reinforce

WHEN TO USE:
Use this when the user sounds happy, relieved, hopeful, proud, or positive.

PRIMARY GOAL:
Reinforce the positive experience and help the user understand what contributed to it.

WHAT TO DO:
- Acknowledge the positive state
- Reinforce it naturally
- Help the user notice what helped
- Encourage awareness of what worked
- Keep the tone warm but not exaggerated

WHAT TO AVOID:
- Do not sound fake or overly enthusiastic
- Do not switch abruptly into advice
- Do not ask shallow questions

GOOD RESPONSE STYLE:
"That sounds like a genuinely positive moment for you. It's good that you're noticing it, because paying attention to what feels better can be really valuable. Reflecting on what contributed to it can help you build more moments like this. What do you think made today feel better for you?"
""",

        "normal_support": """
STRATEGY: normal_support

WHEN TO USE:
Use this when the emotional state is neutral, mixed, or unclear.

PRIMARY GOAL:
Keep the conversation open, supportive, and gently reflective.

WHAT TO DO:
- Acknowledge the situation calmly
- Show light empathy
- Offer one supportive reflection
- Ask one clear open-ended question

WHAT TO AVOID:
- Do not over-dramatize
- Do not force emotional language if it is not present
- Do not become generic

GOOD RESPONSE STYLE:
"It sounds like this situation is taking up some space in your mind right now. Even when feelings are not fully clear yet, it's still useful to slow down and look at what is going on underneath. We can take it one step at a time from here. What feels most important about this right now?"
"""
    }

    few_shots = """
FEW-SHOT EXAMPLES

Example 1
User: I feel anxious about my future.
Assistant: It sounds like you're feeling a lot of uncertainty about what lies ahead. That kind of anxiety can feel really heavy, especially when your mind keeps jumping forward. Sometimes it helps to focus on one part of the future that feels most immediate instead of trying to solve everything at once. What part of your future feels the most uncertain right now?

Example 2
User: I feel sad and alone.
Assistant: It sounds like you're feeling deeply alone right now. That can be a very painful feeling to carry, and it makes sense that it would weigh on you emotionally. Sometimes the hardest part is holding it all by yourself without feeling understood. What has been making you feel the most alone lately?

Example 3
User: I am angry at my manager.
Assistant: It sounds like something in that situation really frustrated you. When something feels unfair or upsetting, anger can come up very quickly, so your reaction makes sense. Taking a moment to understand what triggered that feeling can help you respond more clearly. What part of the interaction affected you the most?

Example 4
User: I am happy today.
Assistant: That sounds like a really positive shift for you. It's good that you're noticing it, because paying attention to these moments can help you understand what supports your wellbeing. Sometimes even one better day can tell us something important about what is helping. What do you think contributed most to feeling good today?

Example 5
User: I feel overwhelmed with work.
Assistant: It sounds like things are feeling really heavy and difficult to manage right now. When too many pressures build up at once, it makes sense to feel overwhelmed rather than clear. Sometimes it helps to slow things down and focus on one manageable piece instead of the whole picture. What part of your workload feels the hardest to carry right now?

Example 6
User: I can't sleep because of stress.
Assistant: It sounds like stress is affecting you both emotionally and physically right now. When the mind stays activated, sleep can become much harder, so what you're experiencing makes sense. Sometimes noticing the main source of pressure can make it easier to calm things down step by step. What feels like the biggest source of stress for you at the moment?

Example 7
User: I feel like I am failing in life.
Assistant: It sounds like you're carrying a lot of pain and self-judgment right now. When someone feels overwhelmed or discouraged, thoughts like this can become very strong, even if they do not tell the whole story. It may help to slow down and look at what has been making this feeling grow lately. What has been making you feel this way most strongly?

Example 8
User: My friend ignored me and now I feel hurt.
Assistant: It sounds like that interaction left you feeling hurt and emotionally unsettled. When someone matters to us, being ignored can feel much more painful than it might seem on the surface. Sometimes these moments affect us because they touch something deeper, like rejection or disconnection. What do you think hurt the most about what happened?
"""

    return (
        base_prompt
        + "\n"
        + strategy_blocks.get(strategy, strategy_blocks["normal_support"])
        + "\n"
        + few_shots
    )

In [ ]:
def enforce_structure(response):
    response = response.strip()

    if not response:
        return "I want to understand what you're going through a bit better. What feels hardest right now?"

    sentences = [s.strip() for s in response.split(".") if s.strip()]

    # keep max 4 sentences
    sentences = sentences[:4]

    if len(sentences) == 0:
        return "I want to understand what you're going through a bit better. What feels hardest right now?"

    # ensure last sentence is a question
    if not sentences[-1].endswith("?"):
        sentences[-1] = sentences[-1].rstrip(".!?") + "?"

    return ". ".join(sentences)

In [ ]:
def safety_filter(response):

    forbidden = [
        "reach out to us",
        "call this number",
        "as a therapist",
        "when I was",
        "in my experience"
    ]

    for f in forbidden:
        if f in response.lower():
            return "It sounds like you're going through something difficult. You're not alone in feeling this way. Taking a moment to reflect on what you're experiencing can help bring clarity. What has been weighing on you the most lately?"

    return response

In [ ]:
def normalize_emotion(label):
    mapping = {
        "fear": "anxiety",
        "sadness": "sadness",
        "anger": "anger",
        "joy": "positive",
        "neutral": "neutral"
    }
    return mapping.get(label, "neutral")

In [ ]:
def choose_strategy(emotion):
    strategy_map = {
        "sadness": "validate_and_support",
        "anxiety": "reassure_and_structure",
        "anger": "deescalate_and_reflect",
        "positive": "encourage_and_reinforce",
        "neutral": "normal_support"
    }
    return strategy_map.get(emotion, "normal_support")

In [ ]:
def get_response_strategy(user_input):
    raw_emotion = detect_emotion(user_input)
    normalized_emotion = normalize_emotion(raw_emotion)
    strategy = choose_strategy(normalized_emotion)

    return {
        "raw_emotion": raw_emotion,
        "emotion": normalized_emotion,
        "strategy": strategy
    }

In [ ]:
def enforce_cbt_structure(response, emotion):

    # fallback template if model is weak
    templates = {
        "anxiety": "It sounds like you're feeling anxious about your situation. It's completely normal to feel this way when things are uncertain. Taking things step by step can help reduce the overwhelm. What part of this situation worries you the most right now?",
        
        "sadness": "It sounds like you're feeling sad and possibly alone. That can be a really heavy feeling to carry. You're not alone in experiencing this, even if it feels that way. What has been weighing on you the most lately?",
        
        "anger": "It sounds like you're feeling frustrated about what happened. That kind of reaction is completely understandable. Taking a moment to reflect can help you respond more calmly. What part of the situation affected you the most?",
        
        "positive": "That sounds like a really positive moment for you. It's great that you're noticing and appreciating it. Reflecting on these moments can help you build more of them. What made this moment feel good for you?",
        
        "neutral": "It sounds like something is on your mind. Taking a moment to reflect can help clarify things. What feels most important about this right now?"
    }

    # if response is too weak → replace it
    if len(response.split()) < 12:
        return templates.get(emotion, templates["neutral"])

    return response

In [ ]:
def generate_response_with_strategy(user_input):
    info = get_response_strategy(user_input)
    strategy = info["strategy"]

    system_prompt = build_system_prompt(strategy)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=110,
            temperature=0.55,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant" in decoded.lower():
        response = decoded.split("assistant")[-1].strip()
    else:
        response = decoded.strip()

    response = safety_filter(response)
    response = enforce_cbt_structure(response, info["emotion"])

    return {
        "raw_emotion": info["raw_emotion"],
        "emotion": info["emotion"],
        "strategy": strategy,
        "system_prompt": system_prompt,
        "response": response
    }

In [ ]:
tests = [
    "I feel anxious about my future",
    "I feel sad and alone",
    "I am angry at my manager",
    "I am happy today"
]

for t in tests:
    result = generate_response_with_strategy(t)

    print("INPUT:", t)
    print("RAW EMOTION:", result["raw_emotion"])
    print("EMOTION:", result["emotion"])
    print("STRATEGY:", result["strategy"])
    print("RESPONSE:", result["response"])
    print("-" * 60)